# Scopus publication-level dataset per author (Notebook)

This notebook builds a **publication-level** dataset for each professor/author using the **Elsevier Scopus Search API** (via `requests`, not `pybliometrics`), which avoids the `service level` issues you hit.

## Output
A tidy table with **one row per publication per author**, including:
- author metadata (author_id, professor name, optional internal id)
- publication metadata (title, year, source/journal, ISSN/eISSN, DOI, subtype, etc.)
- impact (citedby-count)
- collaboration signals (coauthor count; country set if available from search payload)
- raw JSON blobs for traceability (optional)

> Q1/Q2 requires an external journal-quartile lookup (SJR/CiteScore). A placeholder section is included.


In [1]:
# If needed, install dependencies:
# !pip install pandas requests openpyxl pyarrow

import os
import time
import random
import string
import json
import unicodedata
import requests
import pandas as pd
from datetime import datetime
from typing import Any, Dict, List, Optional

## 1) Configuration

In [2]:
# Put your key in an environment variable (recommended):
# export ELSEVIER_API_KEY="..."
API_KEY = os.getenv("ELSEVIER_API_KEY", "***REMOVED***")
assert API_KEY, "Missing ELSEVIER_API_KEY env var"

# Optional: institutional token can unlock richer author/affiliation payloads.
# export ELSEVIER_INSTTOKEN="..."
INST_TOKEN = os.getenv("ELSEVIER_INSTTOKEN")

SEARCH_URL = "https://api.elsevier.com/content/search/scopus"
ABSTRACT_URL_EID = "https://api.elsevier.com/content/abstract/eid"
ABSTRACT_URL_SCOPUS_ID = "https://api.elsevier.com/content/abstract/scopus_id"

# Preferred abstract view. If unauthorized, code falls back automatically.
ABSTRACT_PREFERRED_VIEW = "FULL"

# Default time window (edit as needed)
START_YEAR = 2019
END_YEAR   = 2025

# Throttling (edit as needed)
SLEEP_MIN = 0.2
SLEEP_MAX = 0.6
DETAIL_SLEEP_MIN = 0.15
DETAIL_SLEEP_MAX = 0.4

# Safety caps (edit as needed)
PER_PAGE = 25           # Scopus Search API typical max is 25
MAX_RECORDS_PER_AUTHOR = 5000  # cap to avoid runaway authors

# Output folder
OUT_DIR = "scopus_out"
DATA_DIR = "data"
os.makedirs(OUT_DIR, exist_ok=True)

print("Config loaded:", datetime.now().isoformat())

Config loaded: 2026-03-09T22:22:13.778907


## 2) Define the professor/author roster
Use Scopus Author IDs whenever possible (recommended).

In [3]:
# Minimal roster example:
# You can also load from an Excel/CSV roster.

authors = pd.DataFrame([
    {"prof_id": "TEC-001", "prof_name": "Molina-Perez, Edmundo", "author_id": "56709531800"},
    {"prof_id": "TEC-002", "prof_name": "Stein, Ernesto H.", "author_id": "7202194915"},
    {"prof_id": "TEC-003", "prof_name": "Bernal-Serrano, Daniel", "author_id": "57215937382"},
    {"prof_id": "TEC-004", "prof_name": "Aguilar-Gomez, Sandra", "author_id": "57205638710"},
    {"prof_id": "TEC-005", "prof_name": "Campos-Rivera, Paola Abril", "author_id": "57190304109"},
    {"prof_id": "TEC-006", "prof_name": "Garcia, Magdalena", "author_id": "60054037400"},
    {"prof_id": "TEC-007", "prof_name": "Duran-Fernandez, Roberto", "author_id": "55349505800"},
    {"prof_id": "TEC-008", "prof_name": "Santos, Miguel Angel", "author_id": "57015987800"},
    {"prof_id": "TEC-009", "prof_name": "Popper, Steven W.", "author_id": "7003635400"},
    {"prof_id": "TEC-010", "prof_name": "Peraza-Mues, Gonzalo G.", "author_id": "54997963100"},
    {"prof_id": "TEC-011", "prof_name": "Sobrino, Fernanda", "author_id": "59573511100"},
    {"prof_id": "TEC-012", "prof_name": "Benavides-Rincon, Guillermina", "author_id": "57216153915"},
    {"prof_id": "TEC-013", "prof_name": "Morales-Arilla, Jose", "author_id": "57282638100"},
    {"prof_id": "TEC-014", "prof_name": "Contreras-Loya, David", "author_id": "55971186800"},
    {"prof_id": "TEC-015", "prof_name": "Ponce-Lopez, Roberto", "author_id": "57205704459"},
    {"prof_id": "TEC-016", "prof_name": "Gomez-Zaldivar, Fernando", "author_id": "57219596780"},
    {"prof_id": "TEC-017", "prof_name": "Syme, James", "author_id": "57216353171"},
    {"prof_id": "TEC-018", "prof_name": "Diaz-Dominguez, Alejandro", "author_id": "55581628300"},
    {"prof_id": "TEC-019", "prof_name": "Silverio-Murillo, Adan", "author_id": "57212534001"},
    {"prof_id": "TEC-020", "prof_name": "Villarreal, Héctor J.", "author_id": "36855894400"},
    {"prof_id": "TEC-021", "prof_name": "De Unánue, Adolfo", "author_id": "56013629800"},
])

authors


,prof_id,prof_name,author_id
0,TEC-001,"Molina-Perez, Edmundo",56709531800
1,TEC-002,"Stein, Ernesto H.",7202194915
2,TEC-003,"Bernal-Serrano, Daniel",57215937382
3,TEC-004,"Aguilar-Gomez, Sandra",57205638710
4,TEC-005,"Campos-Rivera, Paola Abril",57190304109
5,TEC-006,"Garcia, Magdalena",60054037400
6,TEC-007,"Duran-Fernandez, Roberto",55349505800
7,TEC-008,"Santos, Miguel Angel",57015987800
8,TEC-009,"Popper, Steven W.",7003635400
9,TEC-010,"Peraza-Mues, Gonzalo G.",54997963100


## 3) Low-level Scopus Search API helpers

In [4]:
def _api_headers() -> Dict[str, str]:
    headers = {"X-ELS-APIKey": API_KEY, "Accept": "application/json"}
    if INST_TOKEN:
        headers["X-ELS-Insttoken"] = INST_TOKEN
    return headers

def _as_list(x: Any) -> List[Any]:
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return [x]

def _clean_text(x: Any) -> Optional[str]:
    if x is None:
        return None
    s = str(x).strip()
    return s or None

def _to_int(x: Any) -> Optional[int]:
    try:
        if x is None or str(x).strip() == "":
            return None
        return int(float(str(x)))
    except Exception:
        return None

def _json_dumps(value: Any) -> Optional[str]:
    if value is None:
        return None
    return json.dumps(value, ensure_ascii=False, separators=(",", ":"))

def _index_to_label(idx: int) -> str:
    # Excel-like labels: a, b, ..., z, aa, ab, ...
    alphabet = string.ascii_lowercase
    idx = int(idx)
    out = ""
    while True:
        idx, rem = divmod(idx, 26)
        out = alphabet[rem] + out
        if idx == 0:
            break
        idx -= 1
    return out

def sleep_a_bit(kind: str = "search"):
    if kind == "detail":
        time.sleep(DETAIL_SLEEP_MIN + random.random() * (DETAIL_SLEEP_MAX - DETAIL_SLEEP_MIN))
    else:
        time.sleep(SLEEP_MIN + random.random() * (SLEEP_MAX - SLEEP_MIN))

def scopus_search(query: str, start: int = 0, count: int = 25, *, timeout: int = 60):
    params = {"query": query, "start": start, "count": count}
    r = requests.get(SEARCH_URL, headers=_api_headers(), params=params, timeout=timeout)
    r.raise_for_status()
    return r.json(), r.headers

def scopus_abstract(
    record_id: str,
    *,
    id_type: str = "eid",
    preferred_view: Optional[str] = ABSTRACT_PREFERRED_VIEW,
    timeout: int = 60,
):
    if id_type == "eid":
        url = f"{ABSTRACT_URL_EID}/{record_id}"
    elif id_type == "scopus_id":
        url = f"{ABSTRACT_URL_SCOPUS_ID}/{record_id}"
    else:
        raise ValueError(f"Unsupported id_type: {id_type}")

    views_to_try = [preferred_view, None] if preferred_view else [None]
    last_error = None

    for view in views_to_try:
        params = {"view": view} if view else {}
        resp = requests.get(url, headers=_api_headers(), params=params, timeout=timeout)

        # If requested view is not allowed, retry with default view.
        if view and resp.status_code in (400, 401):
            last_error = RuntimeError(
                f"Abstract view '{view}' unavailable for {id_type}={record_id}: "
                f"{resp.status_code} {resp.text[:180]}"
            )
            continue

        try:
            resp.raise_for_status()
            return resp.json(), {"view_used": view or "DEFAULT", "status_code": resp.status_code}
        except Exception as exc:
            last_error = exc
            break

    if last_error:
        raise last_error
    raise RuntimeError(f"Unable to retrieve abstract for {id_type}={record_id}")

def build_author_query(author_id: str, start_year: int, end_year: int) -> str:
    # Inclusive years
    return f"AU-ID({author_id}) AND PUBYEAR > {start_year-1} AND PUBYEAR < {end_year+1}"

## 4) Fetch all publications for one author (paged)

In [5]:
def fetch_author_publications(author_id: str, start_year: int, end_year: int,
                              per_page: int = PER_PAGE, max_records: int = MAX_RECORDS_PER_AUTHOR):
    query = build_author_query(author_id, start_year, end_year)

    # First request to get total
    js, _ = scopus_search(query, start=0, count=1)
    total = _to_int((js.get("search-results") or {}).get("opensearch:totalResults")) or 0
    capped_total = min(total, max_records)

    entries_all = []
    for start in range(0, capped_total, per_page):
        js, _ = scopus_search(query, start=start, count=per_page)
        entries = (js.get("search-results") or {}).get("entry", [])
        entries_all.extend(entries)
        sleep_a_bit("search")

    return entries_all, {"total": total, "capped_total": capped_total, "query": query}

## 5) Normalize entries to a publication-level table

In [6]:

def normalize_entries(entries: list) -> pd.DataFrame:
    if not entries:
        return pd.DataFrame()
    df = pd.json_normalize(entries)

    wanted = [
        "dc:identifier",           # SCOPUS_ID
        "eid",
        "dc:title",
        "dc:creator",              # primer autor (string)
        "prism:coverDate",
        "prism:publicationName",
        "prism:issn",
        "prism:eIssn",
        "prism:doi",
        "subtype",
        "subtypeDescription",
        "citedby-count",
        "prism:aggregationType",
        "prism:url",
        "openaccess",
        "openaccessFlag",
        "authkeywords",
        "affiliation",             # raw list – para estadísticas de países
        "author-count",
    ]
    keep = [c for c in wanted if c in df.columns]
    df = df[keep].copy()

    if "prism:coverDate" in df.columns:
        df["year"] = pd.to_datetime(df["prism:coverDate"], errors="coerce").dt.year

    if "citedby-count" in df.columns:
        df["citedby-count"] = pd.to_numeric(df["citedby-count"], errors="coerce")

    if "author-count" in df.columns:
        def _ac(x):
            if isinstance(x, dict):
                return pd.to_numeric(x.get("@total") or x.get("$"), errors="coerce")
            return pd.to_numeric(x, errors="coerce")
        df["coauthor_count"] = df["author-count"].apply(_ac)
        df.drop(columns=["author-count"], inplace=True, errors="ignore")

    if "affiliation" in df.columns:
        def _countries(aff):
            c = [a.get("affiliation-country") for a in _as_list(aff)
                 if isinstance(a, dict) and a.get("affiliation-country")]
            return "; ".join(sorted(set(c))) or None

        def _aff_names(aff):
            n = [a.get("affilname") for a in _as_list(aff)
                 if isinstance(a, dict) and a.get("affilname")]
            return "; ".join(sorted(set(n))) or None

        df["affiliation_countries"] = df["affiliation"].apply(_countries)
        df["affiliation_names"]     = df["affiliation"].apply(_aff_names)
        df.drop(columns=["affiliation"], inplace=True, errors="ignore")

    if "dc:identifier" in df.columns:
        df["scopus_id"] = df["dc:identifier"].astype(str).str.replace("SCOPUS_ID:", "", regex=False)

    return df


# ── Column schemas ─────────────────────────────────────────────────────────────
AUTHOR_AFF_COLUMNS = [
    "author_id", "prof_id", "prof_name",
    "scopus_id", "eid", "dc:title", "prism:coverDate", "year",
    "prism:doi", "prism:publicationName",
    "paper_author_order", "paper_author_auid",
    "paper_author_name", "paper_author_surname",
    "paper_author_given_name", "paper_author_initials",
    "paper_author_orcid",
    "paper_author_affiliation_ids", "paper_author_affiliation_labels",
    "paper_author_raw_affiliations",
    "is_corresponding_author", "is_target_author",
    "affiliation_id", "affiliation_label",
    "affiliation_name", "affiliation_city",
    "affiliation_country", "affiliation_text",
]

DETAIL_LOG_COLUMNS = [
    "cache_key", "eid", "scopus_id",
    "author_detail_view_used", "author_detail_id_type",
    "author_detail_has_authors", "author_detail_warning",
]


# ── Utilidades de normalización ────────────────────────────────────────────────
def _strip_accents(s: str) -> str:
    return "".join(
        c for c in unicodedata.normalize("NFD", s)
        if unicodedata.category(c) != "Mn"
    )

def _normalize_name(name: Optional[str]) -> str:
    """Extrae el apellido normalizado para comparación fuzzy."""
    if not name:
        return ""
    n = _strip_accents(name).lower().strip()
    # Formato 'Apellido, Nombre' o 'Nombre Apellido'
    if "," in n:
        return n.split(",")[0].strip()
    parts = n.split()
    return parts[-1] if parts else n

def _is_same_author(prof_name: Optional[str], oa_display_name: Optional[str]) -> bool:
    """True si los nombres corresponden al mismo autor (por apellido normalizado)."""
    if not prof_name or not oa_display_name:
        return False
    ps = _normalize_name(prof_name)
    os_ = _normalize_name(oa_display_name)
    return bool(ps and os_ and (ps in os_ or os_ in ps))


# ── OpenAlex API ───────────────────────────────────────────────────────────────
# OpenAlex es una base de datos académica abierta (gratuita, sin credenciales).
# Proporciona datos completos de autores, afiliaciones y autor de correspondencia.
OPENALEX_BASE = "https://api.openalex.org"
OPENALEX_HEADERS = {
    "User-Agent": "bibliometrics-research/1.0 (mailto:research@example.com)"
}

def _fetch_openalex_work(doi: Optional[str], timeout: int = 30) -> Optional[Dict[str, Any]]:
    """Consulta OpenAlex por DOI y retorna el objeto 'work' o None si no encuentra."""
    if not doi:
        return None
    doi_clean = str(doi).strip().lstrip("https://doi.org/").lstrip("http://dx.doi.org/")
    if not doi_clean:
        return None
    try:
        url = f"{OPENALEX_BASE}/works/doi:{doi_clean}"
        r = requests.get(url, headers=OPENALEX_HEADERS, timeout=timeout)
        if r.status_code == 404:
            return None
        r.raise_for_status()
        return r.json()
    except Exception:
        return None

def _parse_author_bundle_from_openalex(work: Dict[str, Any]) -> Dict[str, Any]:
    """
    Extrae el bundle completo de autores y afiliaciones desde una respuesta de OpenAlex.

    OpenAlex provee:
    - Lista completa de autores en orden correcto
    - Afiliaciones individuales por autor (instituciones)
    - Raw affiliation strings (texto exacto del paper)
    - Flag de autor de correspondencia (is_corresponding)
    - ORCID de cada autor
    """
    if not work:
        return {"authors": [], "affiliations": [], "affiliation_by_id": {}, "paper_cols": {}}

    authorships: List[Dict[str, Any]] = work.get("authorships") or []

    # ── 1. Construir índice de afiliaciones (instituciones únicas) ─────────
    affiliations: List[Dict[str, Any]] = []
    affiliation_by_id: Dict[str, Any] = {}
    seen_inst: set = set()

    for auth in authorships:
        for inst in (auth.get("institutions") or []):
            inst_id   = _clean_text(inst.get("id"))
            inst_name = _clean_text(inst.get("display_name"))
            country   = _clean_text(inst.get("country_code"))
            inst_type = _clean_text(inst.get("type"))
            aff_text  = ", ".join([x for x in [inst_name, country] if x]) or None

            key = (inst_id or inst_name)
            if not key or key in seen_inst:
                continue
            seen_inst.add(key)

            label = _index_to_label(len(affiliations))
            aff_rec = {
                "affiliation_id":      inst_id,
                "affiliation_label":   label,
                "affiliation_name":    inst_name,
                "affiliation_city":    None,   # OpenAlex no incluye ciudad en authorships
                "affiliation_country": country,
                "affiliation_text":    aff_text,
            }
            affiliations.append(aff_rec)
            if inst_id:
                affiliation_by_id[inst_id] = aff_rec

    # ── 2. Parsear autores ─────────────────────────────────────────────────
    # OpenAlex usa 'author_position': first | middle | last
    # Los autores vienen en el orden correcto del paper
    authors: List[Dict[str, Any]] = []

    for pos, auth in enumerate(authorships, start=1):
        author_obj = auth.get("author") or {}
        oa_id      = _clean_text(author_obj.get("id"))
        name       = _clean_text(author_obj.get("display_name"))
        orcid      = _clean_text(author_obj.get("orcid"))
        is_corr    = bool(auth.get("is_corresponding", False))
        raw_affs   = auth.get("raw_affiliation_strings") or []

        inst_ids = [
            _clean_text(i.get("id"))
            for i in (auth.get("institutions") or [])
            if i.get("id")
        ]
        aff_labels = [
            affiliation_by_id[aid]["affiliation_label"]
            for aid in inst_ids
            if aid in affiliation_by_id
        ]

        # OpenAlex no separa apellido/nombre → dejamos surname/given_name en None
        # pero guardamos el nombre completo en author_name
        authors.append({
            "author_order":              pos,
            "author_auid":               oa_id,   # OpenAlex ID (no Scopus AUID)
            "author_name":               name,
            "author_surname":            None,
            "author_given_name":         None,
            "author_initials":           None,
            "author_orcid":              orcid,
            "author_affiliation_ids":    inst_ids,
            "author_affiliation_labels": aff_labels,
            "is_corresponding_author":   is_corr,
            "raw_affiliation_strings":   raw_affs,
        })

    for a in authors:
        aff_marks = f" ({','.join(a['author_affiliation_labels'])})" if a["author_affiliation_labels"] else ""
        corr_mark = " [corresponding]" if a["is_corresponding_author"] else ""
        a["author_display"] = f"{a['author_name'] or 'Unknown'}{aff_marks}{corr_mark}"

    corr_names_out = [
        a["author_name"] for a in authors
        if a["is_corresponding_author"] and a.get("author_name")
    ]
    affiliation_display = [
        f"{a['affiliation_label']}: {a['affiliation_text']}"
        for a in affiliations if a.get("affiliation_text")
    ]

    return {
        "authors":           authors,
        "affiliations":      affiliations,
        "affiliation_by_id": affiliation_by_id,
        "paper_cols": {
            "paper_author_count_detailed":  len(authors) if authors else None,
            "paper_author_names_ordered":   "; ".join([a["author_name"] for a in authors if a.get("author_name")]) or None,
            "paper_authors":                "; ".join([a["author_display"] for a in authors]) or None,
            "paper_corresponding_authors":  "; ".join(corr_names_out) or None,
            "paper_affiliations":           "; ".join(affiliation_display) or None,
            "paper_authors_json":           _json_dumps(authors),
            "paper_affiliations_json":      _json_dumps(affiliations),
        },
    }


# ── Scopus Abstract API parser (vista FULL) ───────────────────────────────────
def _extract_affiliations(abstract_root: Dict[str, Any]) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    seen: set = set()
    for aff in _as_list(abstract_root.get("affiliation")):
        if not isinstance(aff, dict):
            continue
        aff_id      = _clean_text(aff.get("@id") or aff.get("id") or aff.get("afid"))
        aff_name    = _clean_text(aff.get("affilname") or aff.get("preferred-name") or aff.get("organization"))
        aff_city    = _clean_text(aff.get("affiliation-city"))
        aff_country = _clean_text(aff.get("affiliation-country"))
        aff_address = _clean_text(aff.get("affiliation-address"))
        aff_text    = ", ".join([x for x in [aff_name, aff_address, aff_city, aff_country] if x]) or None
        key = (aff_id, aff_text)
        if key in seen:
            continue
        seen.add(key)
        rows.append({
            "affiliation_id": aff_id, "affiliation_name": aff_name,
            "affiliation_city": aff_city, "affiliation_country": aff_country,
            "affiliation_text": aff_text,
        })
    for i, row in enumerate(rows):
        row["affiliation_label"] = _index_to_label(i)
    return rows

def _extract_correspondence_targets(abstract_root: Dict[str, Any]):
    corr_names: set = set()
    corr_auids: set = set()
    head = (((abstract_root.get("item") or {}).get("bibrecord") or {}).get("head") or {})
    for corr in _as_list(head.get("correspondence")):
        if not isinstance(corr, dict):
            continue
        for person in (_as_list(corr.get("person")) or [corr]):
            if not isinstance(person, dict):
                continue
            indexed_name = _clean_text(person.get("ce:indexed-name") or person.get("indexed-name"))
            surname      = _clean_text(person.get("ce:surname") or person.get("surname"))
            given_name   = _clean_text(person.get("ce:given-name") or person.get("given-name"))
            auid         = _clean_text(person.get("@auid") or person.get("auid"))
            if indexed_name: corr_names.add(indexed_name.lower())
            if surname and given_name: corr_names.add(f"{surname}, {given_name}".lower())
            if surname: corr_names.add(surname.lower())
            if auid: corr_auids.add(auid)
    return corr_names, corr_auids

def _extract_author_affiliation_ids(author_obj: Dict[str, Any]) -> List[str]:
    ids: List[str] = []
    seen: set = set()
    for aff in _as_list(author_obj.get("affiliation")):
        aff_id = _clean_text(aff.get("@id") or aff.get("id") or aff.get("afid")) if isinstance(aff, dict) else _clean_text(aff)
        if aff_id and aff_id not in seen:
            seen.add(aff_id)
            ids.append(aff_id)
    return ids

def parse_author_affiliation_bundle(abstract_json: Dict[str, Any]) -> Dict[str, Any]:
    """Parsea autores y afiliaciones desde la respuesta del Scopus Abstract API (vista FULL)."""
    root = (abstract_json or {}).get("abstracts-retrieval-response") or {}
    affiliations = _extract_affiliations(root)
    affiliation_by_id = {a["affiliation_id"]: a for a in affiliations if a.get("affiliation_id")}
    corr_names, corr_auids = _extract_correspondence_targets(root)

    authors_raw: List[Any] = []
    authors_container = root.get("authors")
    if isinstance(authors_container, dict):
        authors_raw = _as_list(authors_container.get("author"))

    authors: List[Dict[str, Any]] = []
    for pos, author in enumerate(authors_raw, start=1):
        if not isinstance(author, dict):
            continue
        order      = _to_int(author.get("@seq")) or pos
        auid       = _clean_text(author.get("@auid") or author.get("auid") or author.get("authid"))
        surname    = _clean_text(author.get("ce:surname") or author.get("surname"))
        given_name = _clean_text(author.get("ce:given-name") or author.get("given-name"))
        initials   = _clean_text(author.get("ce:initials") or author.get("initials"))
        indexed_nm = _clean_text(author.get("ce:indexed-name") or author.get("indexed-name"))
        author_name = indexed_nm or (f"{surname}, {given_name}" if surname and given_name else surname or given_name or None)

        aff_ids    = _extract_author_affiliation_ids(author)
        aff_labels = [affiliation_by_id[aid]["affiliation_label"] for aid in aff_ids if aid in affiliation_by_id and affiliation_by_id[aid].get("affiliation_label")]
        role_text  = " ".join([str(author.get("ce:role") or ""), str(author.get("@role") or "")]).lower()
        corr_flag  = str(author.get("@corresponding-author") or author.get("corresponding-author") or "").strip().lower() in {"y", "yes", "1", "true"}
        name_vars  = {x.lower() for x in [author_name, f"{surname}, {given_name}" if surname and given_name else None, surname] if x}
        is_corr    = corr_flag or ("correspond" in role_text) or (auid in corr_auids if auid else False) or bool(name_vars.intersection(corr_names))

        authors.append({
            "author_order": order, "author_auid": auid, "author_name": author_name,
            "author_surname": surname, "author_given_name": given_name, "author_initials": initials,
            "author_orcid": None, "author_affiliation_ids": aff_ids, "author_affiliation_labels": aff_labels,
            "is_corresponding_author": bool(is_corr), "raw_affiliation_strings": [],
        })

    authors = sorted(authors, key=lambda x: x["author_order"])
    for a in authors:
        aff_marks = f" ({','.join(a['author_affiliation_labels'])})" if a["author_affiliation_labels"] else ""
        corr_mark = " [corresponding]" if a["is_corresponding_author"] else ""
        a["author_display"] = f"{a['author_name'] or 'Unknown'}{aff_marks}{corr_mark}"

    corr_names_out = [a["author_name"] for a in authors if a["is_corresponding_author"] and a.get("author_name")]
    affiliation_display = [f"{a['affiliation_label']}: {a['affiliation_text']}" for a in affiliations if a.get("affiliation_text")]

    return {
        "authors": authors, "affiliations": affiliations, "affiliation_by_id": affiliation_by_id,
        "paper_cols": {
            "paper_author_count_detailed": len(authors) if authors else None,
            "paper_author_names_ordered": "; ".join([a["author_name"] for a in authors if a.get("author_name")]) or None,
            "paper_authors": "; ".join([a["author_display"] for a in authors]) or None,
            "paper_corresponding_authors": "; ".join(corr_names_out) or None,
            "paper_affiliations": "; ".join(affiliation_display) or None,
            "paper_authors_json": _json_dumps(authors), "paper_affiliations_json": _json_dumps(affiliations),
        },
    }


# ── Orquestador principal ──────────────────────────────────────────────────────
def fetch_paper_author_details(
    eid: Optional[str],
    scopus_id: Optional[str],
    doi: Optional[str],
    details_cache: Dict[str, Dict[str, Any]],
    details_events: List[Dict[str, Any]],
) -> Dict[str, Any]:
    """
    Obtiene datos completos de autores y afiliaciones para un paper.

    Cadena de prioridad:
    1. Scopus Abstract API (vista FULL)  → requiere acceso institucional
    2. OpenAlex API (vía DOI)            → gratuito, completo, incluye autor de correspondencia
    3. Sin datos                         → retorna bundle vacío con warning
    """
    cache_key = _clean_text(scopus_id) or _clean_text(eid)

    _empty_cols: Dict[str, Any] = {
        "paper_author_count_detailed":  None,
        "paper_author_names_ordered":   None,
        "paper_authors":                None,
        "paper_corresponding_authors":  None,
        "paper_affiliations":           None,
        "paper_authors_json":           None,
        "paper_affiliations_json":      None,
        "author_detail_view_used":      None,
        "author_detail_id_type":        None,
        "author_detail_has_authors":    False,
        "author_detail_warning":        "Missing EID/Scopus ID",
    }

    if not cache_key:
        return {"authors": [], "affiliations": [], "affiliation_by_id": {}, "paper_cols": _empty_cols}

    if cache_key in details_cache:
        return details_cache[cache_key]

    bundle: Optional[Dict[str, Any]] = None
    last_error = None

    # ── 1. Scopus Abstract API ─────────────────────────────────────────────
    id_candidates = []
    if _clean_text(eid):       id_candidates.append(("eid",       _clean_text(eid)))
    if _clean_text(scopus_id): id_candidates.append(("scopus_id", _clean_text(scopus_id)))

    for id_type, record_id in id_candidates:
        try:
            abstract_json, meta = scopus_abstract(record_id, id_type=id_type)
            parsed = parse_author_affiliation_bundle(abstract_json)
            if parsed["authors"]:
                paper_cols = dict(parsed["paper_cols"])
                paper_cols["author_detail_view_used"]   = meta.get("view_used")
                paper_cols["author_detail_id_type"]     = id_type
                paper_cols["author_detail_has_authors"] = True
                paper_cols["author_detail_warning"]     = None
                bundle = {
                    "authors": parsed["authors"], "affiliations": parsed["affiliations"],
                    "affiliation_by_id": parsed["affiliation_by_id"], "paper_cols": paper_cols,
                }
                sleep_a_bit("detail")
                break
        except Exception as exc:
            last_error = exc

    # ── 2. OpenAlex API (fallback gratuito) ────────────────────────────────
    if bundle is None and _clean_text(doi):
        try:
            oa_work = _fetch_openalex_work(_clean_text(doi))
            if oa_work:
                oa_bundle = _parse_author_bundle_from_openalex(oa_work)
                if oa_bundle["authors"]:
                    paper_cols = dict(oa_bundle["paper_cols"])
                    paper_cols["author_detail_view_used"]   = "OPENALEX"
                    paper_cols["author_detail_id_type"]     = "doi"
                    paper_cols["author_detail_has_authors"] = True
                    paper_cols["author_detail_warning"]     = (
                        "Authors from OpenAlex (Scopus FULL view requires institutional access). "
                        "Author IDs are OpenAlex IDs, not Scopus AUIDs."
                    )
                    bundle = {
                        "authors": oa_bundle["authors"], "affiliations": oa_bundle["affiliations"],
                        "affiliation_by_id": oa_bundle["affiliation_by_id"], "paper_cols": paper_cols,
                    }
                    time.sleep(DETAIL_SLEEP_MIN)  # polite rate-limiting for OpenAlex
        except Exception as exc:
            last_error = exc

    # ── 3. Sin datos ───────────────────────────────────────────────────────
    if bundle is None:
        msg = str(last_error)[:280] if last_error else "No author data available"
        cols = dict(_empty_cols)
        cols["author_detail_warning"] = msg
        bundle = {"authors": [], "affiliations": [], "affiliation_by_id": {}, "paper_cols": cols}

    details_cache[cache_key] = bundle
    details_events.append({
        "cache_key":                  cache_key,
        "eid":                        _clean_text(eid),
        "scopus_id":                  _clean_text(scopus_id),
        "author_detail_view_used":    bundle["paper_cols"].get("author_detail_view_used"),
        "author_detail_id_type":      bundle["paper_cols"].get("author_detail_id_type"),
        "author_detail_has_authors":  bundle["paper_cols"].get("author_detail_has_authors"),
        "author_detail_warning":      bundle["paper_cols"].get("author_detail_warning"),
    })
    return bundle


# ── Enriquecer DataFrame con autores y afiliaciones ───────────────────────────
def enrich_with_author_details(
    df: pd.DataFrame,
    details_cache: Dict[str, Dict[str, Any]],
    details_events: List[Dict[str, Any]],
):
    if df.empty:
        return df.copy(), []

    paper_detail_rows: List[Dict[str, Any]] = []
    author_aff_rows: List[Dict[str, Any]] = []

    for _, row in df.iterrows():
        eid       = row.get("eid")
        scopus_id = row.get("scopus_id")
        doi       = row.get("prism:doi")
        prof_name = row.get("prof_name")

        bundle = fetch_paper_author_details(eid, scopus_id, doi, details_cache, details_events)
        paper_detail_rows.append(bundle["paper_cols"])

        for pa in bundle["authors"]:
            aff_ids = pa.get("author_affiliation_ids") or []

            base = {
                "author_id":                       row.get("author_id"),
                "prof_id":                         row.get("prof_id"),
                "prof_name":                       prof_name,
                "scopus_id":                       scopus_id,
                "eid":                             eid,
                "dc:title":                        row.get("dc:title"),
                "prism:coverDate":                 row.get("prism:coverDate"),
                "year":                            row.get("year"),
                "prism:doi":                       doi,
                "prism:publicationName":           row.get("prism:publicationName"),
                "paper_author_order":              pa.get("author_order"),
                "paper_author_auid":               pa.get("author_auid"),
                "paper_author_name":               pa.get("author_name"),
                "paper_author_surname":            pa.get("author_surname"),
                "paper_author_given_name":         pa.get("author_given_name"),
                "paper_author_initials":           pa.get("author_initials"),
                "paper_author_orcid":              pa.get("author_orcid"),
                "paper_author_affiliation_ids":    "; ".join(aff_ids) or None,
                "paper_author_affiliation_labels": "; ".join(pa.get("author_affiliation_labels") or []) or None,
                "paper_author_raw_affiliations":   "; ".join(pa.get("raw_affiliation_strings") or []) or None,
                "is_corresponding_author":         pa.get("is_corresponding_author"),
                # Comparamos por Scopus AUID (Scopus FULL view) o por nombre (OpenAlex)
                "is_target_author": (
                    str(row.get("author_id")) == str(pa.get("author_auid"))
                    if pa.get("author_auid") and not str(pa.get("author_auid", "")).startswith("https://")
                    else _is_same_author(prof_name, pa.get("author_name"))
                ),
            }

            if not aff_ids:
                author_aff_rows.append({
                    **base,
                    "affiliation_id": None, "affiliation_label": None,
                    "affiliation_name": None, "affiliation_city": None,
                    "affiliation_country": None, "affiliation_text": None,
                })
            else:
                for aid in aff_ids:
                    aff = bundle["affiliation_by_id"].get(aid, {})
                    author_aff_rows.append({
                        **base,
                        "affiliation_id":      aid,
                        "affiliation_label":   aff.get("affiliation_label"),
                        "affiliation_name":    aff.get("affiliation_name"),
                        "affiliation_city":    aff.get("affiliation_city"),
                        "affiliation_country": aff.get("affiliation_country"),
                        "affiliation_text":    aff.get("affiliation_text"),
                    })

    details_df  = pd.DataFrame(paper_detail_rows)
    enriched_df = pd.concat([df.reset_index(drop=True), details_df], axis=1)
    return enriched_df, author_aff_rows


## 6) Build the final dataset (one row per publication per author)

In [7]:
def build_publication_level_dataset(authors_df: pd.DataFrame, start_year: int, end_year: int):
    out_frames = []
    logs = []
    author_aff_rows_all = []
    details_cache: Dict[str, Dict[str, Any]] = {}
    details_events: List[Dict[str, Any]] = []

    for _, row in authors_df.iterrows():
        author_id = str(row["author_id"])
        prof_id = row.get("prof_id", None)
        prof_name = row.get("prof_name", None)

        print(f"Fetching AU-ID({author_id}) {start_year}-{end_year} ...")
        entries, meta = fetch_author_publications(author_id, start_year, end_year)
        logs.append({"author_id": author_id, "prof_id": prof_id, "prof_name": prof_name, **meta})

        df = normalize_entries(entries)
        if df.empty:
            continue

        # Add tracked-author dimensions
        df.insert(0, "author_id", author_id)
        df.insert(1, "prof_id", prof_id)
        df.insert(2, "prof_name", prof_name)

        # Enrich every paper with full author/affiliation details (when entitlement allows it)
        df, author_aff_rows = enrich_with_author_details(df, details_cache, details_events)
        author_aff_rows_all.extend(author_aff_rows)
        out_frames.append(df)

    # Save fetch log
    log_df = pd.DataFrame(logs)
    log_path = os.path.join(OUT_DIR, "fetch_log.csv")
    log_df.to_csv(log_path, index=False)
    print("Saved fetch log:", log_path)

    # Save abstract detail status log
    details_log_df = pd.DataFrame(details_events, columns=DETAIL_LOG_COLUMNS)
    details_log_path = os.path.join(OUT_DIR, "abstract_detail_log.csv")
    details_log_df.to_csv(details_log_path, index=False)
    print("Saved abstract detail log:", details_log_path)

    if not out_frames:
        empty_authors_df = pd.DataFrame(author_aff_rows_all, columns=AUTHOR_AFF_COLUMNS)
        return pd.DataFrame(), empty_authors_df, details_log_df

    final = pd.concat(out_frames, ignore_index=True)
    paper_authors_df = pd.DataFrame(author_aff_rows_all, columns=AUTHOR_AFF_COLUMNS)
    return final, paper_authors_df, details_log_df


pubs, paper_authors, abstract_details_log = build_publication_level_dataset(authors, START_YEAR, END_YEAR)
pubs.shape, pubs.head(3)

Fetching AU-ID(56709531800) 2019-2025 ...
Fetching AU-ID(7202194915) 2019-2025 ...
Fetching AU-ID(57215937382) 2019-2025 ...
Fetching AU-ID(57205638710) 2019-2025 ...
Fetching AU-ID(57190304109) 2019-2025 ...
Fetching AU-ID(60054037400) 2019-2025 ...
Fetching AU-ID(55349505800) 2019-2025 ...
Fetching AU-ID(57015987800) 2019-2025 ...
Fetching AU-ID(7003635400) 2019-2025 ...
Fetching AU-ID(54997963100) 2019-2025 ...
Fetching AU-ID(59573511100) 2019-2025 ...
Fetching AU-ID(57216153915) 2019-2025 ...
Fetching AU-ID(57282638100) 2019-2025 ...
Fetching AU-ID(55971186800) 2019-2025 ...
Fetching AU-ID(57205704459) 2019-2025 ...
Fetching AU-ID(57219596780) 2019-2025 ...
Fetching AU-ID(57216353171) 2019-2025 ...
Fetching AU-ID(55581628300) 2019-2025 ...
Fetching AU-ID(57212534001) 2019-2025 ...
Fetching AU-ID(36855894400) 2019-2025 ...
Fetching AU-ID(56013629800) 2019-2025 ...
Saved fetch log: scopus_out/fetch_log.csv
Saved abstract detail log: scopus_out/abstract_detail_log.csv


((167, 34),
      author_id  prof_id              prof_name           dc:identifier  \
 0  56709531800  TEC-001  Molina-Perez, Edmundo  SCOPUS_ID:105015077202   
 1  56709531800  TEC-001  Molina-Perez, Edmundo  SCOPUS_ID:105013320148   
 2  56709531800  TEC-001  Molina-Perez, Edmundo   SCOPUS_ID:85185156225   
 
                    eid                                           dc:title  \
 0  2-s2.0-105015077202  Groundwater parameters estimation: A hybrid ap...   
 1  2-s2.0-105013320148  Using a robust decision making (RDM) approach ...   
 2   2-s2.0-85185156225  The economics of decarbonizing Costa Rica's ag...   
 
     dc:creator prism:coverDate  \
 0     Tesen K.      2025-12-09   
 1   Poblete D.      2025-01-01   
 2  Banerjee O.      2024-04-01   
 
                                prism:publicationName prism:issn  ...  \
 0  Engineering Applications of Artificial Intelli...   09521976  ...   
 1                 Frontiers in Environmental Science        NaN  ...   
 2         

## 6b) Prueba rápida: verificar extracción de autores en 1 paper

Antes de ejecutar el dataset completo, se valida que el parsing desde el Search API funciona correctamente con un paper de ejemplo (Molina-Perez, Edmundo – TEC-001).

In [8]:

# ── Prueba con el paper de ejemplo del usuario ─────────────────────────────────
# Paper: "Groundwater parameters estimation: A hybrid approach..."
# Autores esperados: Tesen(a,b,c) [corresponding], Cortés(d), Vicuña(a,e),
#                   Molina-Perez(d), Suárez(a,c,f)

TEST_DOI = "10.1016/j.engappai.2025.112098"

print("=== Probando OpenAlex para el paper de ejemplo ===\n")
oa_work = _fetch_openalex_work(TEST_DOI)
if oa_work:
    bundle = _parse_author_bundle_from_openalex(oa_work)
    print(f"Título:  {oa_work.get('title')}")
    print(f"Fuente de datos: OpenAlex")
    print(f"\nAutores ({len(bundle['authors'])}):")
    for a in bundle["authors"]:
        print(f"  [{a['author_order']}] {a['author_name']}")
        print(f"       ORCID: {a['author_orcid'] or 'N/A'}")
        print(f"       Afiliaciones ({a['author_affiliation_labels']}): {a['author_affiliation_ids']}")
        print(f"       Correspondencia: {a['is_corresponding_author']}")
        for raw in (a.get('raw_affiliation_strings') or []):
            print(f"       Raw aff: {raw}")
    print(f"\nAfiliaciones únicas del paper ({len(bundle['affiliations'])}):")
    for af in bundle["affiliations"]:
        print(f"  ({af['affiliation_label']}) {af['affiliation_name']} | {af['affiliation_country']}")
    print(f"\n── paper_authors ──")
    print(bundle["paper_cols"]["paper_authors"])
    print(f"\n── paper_affiliations ──")
    print(bundle["paper_cols"]["paper_affiliations"])
    print(f"\n── paper_corresponding_authors ──")
    print(bundle["paper_cols"]["paper_corresponding_authors"])
else:
    print("ERROR: No se pudo obtener datos de OpenAlex para este DOI")


=== Probando OpenAlex para el paper de ejemplo ===

Título:  Groundwater parameters estimation: A hybrid approach of convolutional layers with asynchronous and distributed bio-inspired algorithms
Fuente de datos: OpenAlex

Autores (5):
  [1] Kiara Tesen
       ORCID: https://orcid.org/0000-0002-1218-4957
       Afiliaciones (['a', 'b', 'c']): ['https://openalex.org/I159360068', 'https://openalex.org/I162148367', 'https://openalex.org/I4210098341']
       Correspondencia: True
       Raw aff: Centro de Desarrollo Urbano Sustentable (CEDEUS), Pontificia Universidad Católica de Chile, Santiago, Chile
       Raw aff: Departamento de Ingeniería Civil, Universidad de Piura, Piura, Peru
       Raw aff: Departamento de Ingeniería Hidráulica y Ambiental, Pontificia Universidad Católica de Chile, Santiago, Chile
  [2] H Monardes Cortés
       ORCID: N/A
       Afiliaciones (['d']): ['https://openalex.org/I98461037']
       Correspondencia: False
       Raw aff: Escuela de Gobierno y Transformaci

In [9]:
pubs.iloc[7]

author_id                                                            56709531800
prof_id                                                                  TEC-001
prof_name                                                  Molina-Perez, Edmundo
dc:identifier                                              SCOPUS_ID:85164252140
eid                                                           2-s2.0-85164252140
dc:title                       Knowledge co-production for decision-making in...
dc:creator                                                         Moallemi E.A.
prism:coverDate                                                       2023-09-01
prism:publicationName                                Global Environmental Change
prism:issn                                                              09593780
prism:eIssn                                                                  NaN
prism:doi                                        10.1016/j.gloenvcha.2023.102727
subtype                     

In [10]:
pubs.head(3)

,author_id,prof_id,prof_name,dc:identifier,eid,dc:title,dc:creator,prism:coverDate,prism:publicationName,prism:issn,...,paper_author_names_ordered,paper_authors,paper_corresponding_authors,paper_affiliations,paper_authors_json,paper_affiliations_json,author_detail_view_used,author_detail_id_type,author_detail_has_authors,author_detail_warning
0,56709531800,TEC-001,"Molina-Perez, Edmundo",SCOPUS_ID:105015077202,2-s2.0-105015077202,Groundwater parameters estimation: A hybrid ap...,Tesen K.,2025-12-09,Engineering Applications of Artificial Intelli...,09521976,...,Kiara Tesen; H Monardes Cortés; Sebastián Vicu...,"Kiara Tesen (a,b,c) [corresponding]; H Monarde...",Kiara Tesen,"a: University of Piura, PE; b: Pontificia Univ...","[{""author_order"":1,""author_auid"":null,""author_...","[{""affiliation_id"":""https://openalex.org/I1593...",OPENALEX,doi,True,Authors from OpenAlex (Scopus FULL view requir...
1,56709531800,TEC-001,"Molina-Perez, Edmundo",SCOPUS_ID:105013320148,2-s2.0-105013320148,Using a robust decision making (RDM) approach ...,Poblete D.,2025-01-01,Frontiers in Environmental Science,NaN,...,David Figueroa Poblete; Sebastián Vicuña; Seba...,"David Figueroa Poblete (a,b) [corresponding]; ...",David Figueroa Poblete,"a: Pontificia Universidad Católica de Chile, C...","[{""author_order"":1,""author_auid"":""https://open...","[{""affiliation_id"":""https://openalex.org/I1621...",OPENALEX,doi,True,Authors from OpenAlex (Scopus FULL view requir...
2,56709531800,TEC-001,"Molina-Perez, Edmundo",SCOPUS_ID:85185156225,2-s2.0-85185156225,The economics of decarbonizing Costa Rica's ag...,Banerjee O.,2024-04-01,Ecological Economics,09218009,...,Onil Banerjee; Martín Cicowiez; Renato Vargas;...,Onil Banerjee [corresponding]; Martín Cicowiez...,Onil Banerjee,"a: Universidad Nacional de La Plata, AR; b: Te...","[{""author_order"":1,""author_auid"":""https://open...","[{""affiliation_id"":""https://openalex.org/I8743...",OPENALEX,doi,True,Authors from OpenAlex (Scopus FULL view requir...


## 7) Journal quartiles
Provide a journal lookup table with ISSN/eISSN -> quartile.


In [11]:
# Example expected schema for journal_quartiles.csv:
# issn, eissn, year, quartile  (quartile in {Q1,Q2,Q3,Q4})
# Load and left-join on issn/eissn + year.

ql = pd.read_csv(os.path.join(DATA_DIR, "scimagojr 2024.csv"), sep=";")
ql.head(3)

,Rank,Sourceid,Title,Type,Issn,Publisher,Open Access,Open Access Diamond,SJR,SJR Best Quartile,...,Ref. / Doc.,%Female,Overton,SDG,Country,Region,Publisher.1,Coverage,Categories,Areas
0,1,28773,Ca-A Cancer Journal for Clinicians,journal,"15424863, 00079235",John Wiley and Sons Inc,No,No,"145,004",Q1,...,"62,88","48,21",4,37,United States,Northern America,John Wiley and Sons Inc,1950-2025,Hematology (Q1); Oncology (Q1),Medicine
1,2,19434,MMWR Recommendations and Reports,journal,"10575987, 15458601",Centers for Disease Control and Prevention (CDC),Yes,No,"41,754",Q1,...,"275,33","75,93",1,5,United States,Northern America,Centers for Disease Control and Prevention (CDC),1990-2024,Epidemiology (Q1); Health Information Manageme...,Environmental Science; Health Professions; Med...
2,3,20315,Nature Reviews Molecular Cell Biology,journal,"14710072, 14710080",Nature Research,No,No,"37,353",Q1,...,"92,45","41,22",0,15,United Kingdom,Western Europe,Nature Research,2000-2025,Cell Biology (Q1); Molecular Biology (Q1),"Biochemistry, Genetics and Molecular Biology"


In [12]:
ql

,Rank,Sourceid,Title,Type,Issn,Publisher,Open Access,Open Access Diamond,SJR,SJR Best Quartile,...,Ref. / Doc.,%Female,Overton,SDG,Country,Region,Publisher.1,Coverage,Categories,Areas
0,1,28773,Ca-A Cancer Journal for Clinicians,journal,"15424863, 00079235",John Wiley and Sons Inc,No,No,"145,004",Q1,...,"62,88","48,21",4,37,United States,Northern America,John Wiley and Sons Inc,1950-2025,Hematology (Q1); Oncology (Q1),Medicine
1,2,19434,MMWR Recommendations and Reports,journal,"10575987, 15458601",Centers for Disease Control and Prevention (CDC),Yes,No,"41,754",Q1,...,"275,33","75,93",1,5,United States,Northern America,Centers for Disease Control and Prevention (CDC),1990-2024,Epidemiology (Q1); Health Information Manageme...,Environmental Science; Health Professions; Med...
2,3,20315,Nature Reviews Molecular Cell Biology,journal,"14710072, 14710080",Nature Research,No,No,"37,353",Q1,...,"92,45","41,22",0,15,United Kingdom,Western Europe,Nature Research,2000-2025,Cell Biology (Q1); Molecular Biology (Q1),"Biochemistry, Genetics and Molecular Biology"
3,4,29431,Quarterly Journal of Economics,journal,"00335533, 15314650",Oxford University Press,No,No,"35,995",Q1,...,"69,79","25,18",35,27,United Kingdom,Western Europe,Oxford University Press,1886-2025,Economics and Econometrics (Q1),"Economics, Econometrics and Finance"
4,5,20425,Nature Reviews Drug Discovery,journal,"14741784, 14741776",Nature Research,No,No,"30,506",Q1,...,"35,66","26,67",1,58,United Kingdom,Western Europe,Nature Research,2002-2025,Drug Discovery (Q1); Medicine (miscellaneous) ...,"Medicine; Pharmacology, Toxicology and Pharmac..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31131,31132,145515,Waves in Random and Complex Media (discontinued),journal,"17455049, 17455030",Taylor and Francis Ltd.,No,No,NaN,-,...,"49,05","25,14",0,20,United Kingdom,Western Europe,Taylor and Francis Ltd.,2005-2024,Engineering (miscellaneous); Physics and Astro...,Engineering; Physics and Astronomy
31132,31133,21101174249,Word and Music Studies,book series,15660958,Brill: Rodopi,No,No,NaN,-,...,"79,44","100,00",0,0,Netherlands,Western Europe,Brill: Rodopi,"1999-2001, 2004-2006, 2008, 2010-2011, 2014-20...",Arts and Humanities (miscellaneous); Literatur...,Arts and Humanities
31133,31134,18033,"World Dredging, Mining and Construction",trade journal,10450343,Placer Management Corp.,No,No,NaN,-,...,"0,00","100,00",0,1,United States,Northern America,Placer Management Corp.,"1979, 1988-2024",Building and Construction; Ocean Engineering; ...,Earth and Planetary Sciences; Engineering
31134,31135,21100898632,World Scientific-Now Publishers Series in Busi...,book series,22513442,World Scientific,No,No,NaN,-,...,"31,33","33,33",0,0,Singapore,Asiatic Region,World Scientific,"2018-2020, 2023-2024","Business, Management and Accounting (miscellan...","Business, Management and Accounting"


In [13]:
ql_long = ql.copy()
ql_long['Issn'] = ql_long['Issn'].str.split(',')
ql_long = ql_long.explode('Issn')
ql_long['Issn'] = ql_long['Issn'].str.strip()

# Keep SJR Best Quartile and Publisher columns
# (adjust column names if they differ in your CSV)
ql_long = ql_long[['Issn', 'SJR Best Quartile', 'Publisher']]
ql_long.head()

,Issn,SJR Best Quartile,Publisher
0,15424863,Q1,John Wiley and Sons Inc
0,00079235,Q1,John Wiley and Sons Inc
1,10575987,Q1,Centers for Disease Control and Prevention (CDC)
1,15458601,Q1,Centers for Disease Control and Prevention (CDC)
2,14710072,Q1,Nature Research


In [14]:

pubs["Issn"] = pubs["prism:issn"].fillna(pubs["prism:eIssn"])
pubs.head(3)

,author_id,prof_id,prof_name,dc:identifier,eid,dc:title,dc:creator,prism:coverDate,prism:publicationName,prism:issn,...,paper_authors,paper_corresponding_authors,paper_affiliations,paper_authors_json,paper_affiliations_json,author_detail_view_used,author_detail_id_type,author_detail_has_authors,author_detail_warning,Issn
0,56709531800,TEC-001,"Molina-Perez, Edmundo",SCOPUS_ID:105015077202,2-s2.0-105015077202,Groundwater parameters estimation: A hybrid ap...,Tesen K.,2025-12-09,Engineering Applications of Artificial Intelli...,09521976,...,"Kiara Tesen (a,b,c) [corresponding]; H Monarde...",Kiara Tesen,"a: University of Piura, PE; b: Pontificia Univ...","[{""author_order"":1,""author_auid"":null,""author_...","[{""affiliation_id"":""https://openalex.org/I1593...",OPENALEX,doi,True,Authors from OpenAlex (Scopus FULL view requir...,09521976
1,56709531800,TEC-001,"Molina-Perez, Edmundo",SCOPUS_ID:105013320148,2-s2.0-105013320148,Using a robust decision making (RDM) approach ...,Poblete D.,2025-01-01,Frontiers in Environmental Science,NaN,...,"David Figueroa Poblete (a,b) [corresponding]; ...",David Figueroa Poblete,"a: Pontificia Universidad Católica de Chile, C...","[{""author_order"":1,""author_auid"":""https://open...","[{""affiliation_id"":""https://openalex.org/I1621...",OPENALEX,doi,True,Authors from OpenAlex (Scopus FULL view requir...,2296665X
2,56709531800,TEC-001,"Molina-Perez, Edmundo",SCOPUS_ID:85185156225,2-s2.0-85185156225,The economics of decarbonizing Costa Rica's ag...,Banerjee O.,2024-04-01,Ecological Economics,09218009,...,Onil Banerjee [corresponding]; Martín Cicowiez...,Onil Banerjee,"a: Universidad Nacional de La Plata, AR; b: Te...","[{""author_order"":1,""author_auid"":""https://open...","[{""affiliation_id"":""https://openalex.org/I8743...",OPENALEX,doi,True,Authors from OpenAlex (Scopus FULL view requir...,09218009


In [15]:
pubs_q = pubs.merge(ql_long, how="left", left_on=["Issn"], right_on=["Issn"])

pubs_q.head(3)

,author_id,prof_id,prof_name,dc:identifier,eid,dc:title,dc:creator,prism:coverDate,prism:publicationName,prism:issn,...,paper_affiliations,paper_authors_json,paper_affiliations_json,author_detail_view_used,author_detail_id_type,author_detail_has_authors,author_detail_warning,Issn,SJR Best Quartile,Publisher
0,56709531800,TEC-001,"Molina-Perez, Edmundo",SCOPUS_ID:105015077202,2-s2.0-105015077202,Groundwater parameters estimation: A hybrid ap...,Tesen K.,2025-12-09,Engineering Applications of Artificial Intelli...,09521976,...,"a: University of Piura, PE; b: Pontificia Univ...","[{""author_order"":1,""author_auid"":null,""author_...","[{""affiliation_id"":""https://openalex.org/I1593...",OPENALEX,doi,True,Authors from OpenAlex (Scopus FULL view requir...,09521976,Q1,Elsevier Ltd
1,56709531800,TEC-001,"Molina-Perez, Edmundo",SCOPUS_ID:105013320148,2-s2.0-105013320148,Using a robust decision making (RDM) approach ...,Poblete D.,2025-01-01,Frontiers in Environmental Science,NaN,...,"a: Pontificia Universidad Católica de Chile, C...","[{""author_order"":1,""author_auid"":""https://open...","[{""affiliation_id"":""https://openalex.org/I1621...",OPENALEX,doi,True,Authors from OpenAlex (Scopus FULL view requir...,2296665X,Q1,Frontiers Media SA
2,56709531800,TEC-001,"Molina-Perez, Edmundo",SCOPUS_ID:85185156225,2-s2.0-85185156225,The economics of decarbonizing Costa Rica's ag...,Banerjee O.,2024-04-01,Ecological Economics,09218009,...,"a: Universidad Nacional de La Plata, AR; b: Te...","[{""author_order"":1,""author_auid"":""https://open...","[{""affiliation_id"":""https://openalex.org/I8743...",OPENALEX,doi,True,Authors from OpenAlex (Scopus FULL view requir...,09218009,Q1,Elsevier B.V.


In [16]:
pubs_q.to_csv(os.path.join(OUT_DIR, "scopus_publications.csv"), index=False)
paper_authors.to_csv(os.path.join(OUT_DIR, "scopus_publication_authors.csv"), index=False)
abstract_details_log.to_csv(os.path.join(OUT_DIR, "abstract_detail_log.csv"), index=False)

print("Saved:")
print("-", os.path.join(OUT_DIR, "scopus_publications.csv"))
print("-", os.path.join(OUT_DIR, "scopus_publication_authors.csv"))
print("-", os.path.join(OUT_DIR, "abstract_detail_log.csv"))

Saved:
- scopus_out/scopus_publications.csv
- scopus_out/scopus_publication_authors.csv
- scopus_out/abstract_detail_log.csv


## 9) Sanity checks

In [17]:
# ── Validaciones básicas ────────────────────────────────────────────────────────
assert pubs["author_id"].notna().all(), "Hay filas sin author_id"
print("Profesores únicos:", pubs["prof_id"].nunique())
print("Años cubiertos:   ", pubs["year"].min(), "-", pubs["year"].max())
print("Total publicaciones:", len(pubs))

# Calidad de datos de autores
detailed = pubs["paper_author_count_detailed"].notna().sum()
print(f"\nPapers con autores detallados: {detailed} de {len(pubs)} ({100*detailed//len(pubs) if len(pubs) else 0}%)")

# Fuentes de datos utilizadas
if "author_detail_view_used" in pubs.columns:
    print("\nFuentes de datos de autores:")
    print(pubs["author_detail_view_used"].value_counts(dropna=False).to_string())

print(f"\nFilas en scopus_publication_authors: {len(paper_authors)}")
if not paper_authors.empty:
    print("Columnas:", list(paper_authors.columns))
    print(f"\nAutores de correspondencia encontrados: {int(paper_authors['is_corresponding_author'].fillna(False).sum())}")
    print(f"Filas con afiliación identificada:     {int(paper_authors['affiliation_name'].notna().sum())}")
    print(f"Filas con ORCID:                       {int(paper_authors['paper_author_orcid'].notna().sum())}")

    # Ejemplo: primer paper del primer profesor con autores
    sample = paper_authors[paper_authors["affiliation_name"].notna()].head(1)
    if not sample.empty:
        row = sample.iloc[0]
        print(f"\n── Ejemplo de fila ──")
        print(f"Paper:     {row.get('dc:title','')[:60]}")
        print(f"Autor:     {row.get('paper_author_name')} (orden {row.get('paper_author_order')})")
        print(f"Afil:      {row.get('affiliation_name')} | {row.get('affiliation_country')}")
        print(f"Corresp.:  {row.get('is_corresponding_author')}")
        print(f"Target:    {row.get('is_target_author')}")

# Publicaciones por profesor
print("\nPublicaciones por profesor (top 10):")
print(pubs.groupby(["prof_id", "prof_name"]).size().sort_values(ascending=False).head(10).to_string())


Profesores únicos: 20
Años cubiertos:    2019 - 2025
Total publicaciones: 167

Papers con autores detallados: 163 de 167 (97%)

Fuentes de datos de autores:
author_detail_view_used
OPENALEX    163
NaN           4

Filas en scopus_publication_authors: 963
Columnas: ['author_id', 'prof_id', 'prof_name', 'scopus_id', 'eid', 'dc:title', 'prism:coverDate', 'year', 'prism:doi', 'prism:publicationName', 'paper_author_order', 'paper_author_auid', 'paper_author_name', 'paper_author_surname', 'paper_author_given_name', 'paper_author_initials', 'paper_author_orcid', 'paper_author_affiliation_ids', 'paper_author_affiliation_labels', 'paper_author_raw_affiliations', 'is_corresponding_author', 'is_target_author', 'affiliation_id', 'affiliation_label', 'affiliation_name', 'affiliation_city', 'affiliation_country', 'affiliation_text']

Autores de correspondencia encontrados: 209
Filas con afiliación identificada:     856
Filas con ORCID:                       793

── Ejemplo de fila ──
Paper:     Grou